## Mission 1

- 미션 1. 신고자 성별 분류 대화를 입력 받아서 (음성) -> 남, 여 분류
  - 음성 파일 및 전사 및 annotation 파일 존재 (json) 각 발화 조각들의 정보들
  - 조건: 외부 데이터 사용 불가, 미션에서 제시하는 입출력만 사용
  - 구글, 지피티 등 API를 사용해서 답을 내면 안됨 (우리가 코드를 짜야함)
  - 서울권 데이터만 사용, validation 폴더는 사용 금지(단 early stopping/체크포인트 선정 등 검증 목적은 허용), 샘플 전처리 금지
  - 음성 처리 가이드 제시 (ppt 참고) -> 문제에 따라 유리한 방식이 다를 것
- References:
  - 성능 평가도 진행
  - 학습 모델과 추론 코드를 함께 제출, inference를 돌릴 수 있는 코드를 제출해야 함
  - 불필요한 코드 넣지 말고, 주석은 필수로 달아주기


### 0. GPU 확인

In [1]:
# 어떤 GPU가 배정됐는지 확인
!nvidia-smi


Fri Sep 11 15:03:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### 1. 환경 설정

In [2]:
# ===== 환경 설정 =====
IS_DRIVE = True # Google Drive 마운트해서 쓰는 중이면 True, 로컬 런타임이면 False

if IS_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

!pip install -q librosa soundfile scikit-learn

import os, json, shutil, random
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score

# 재현성을 위한 시드 고정
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}, GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

# 데이터 루트: Drive 마운트 여부에 따라 자동 결정
DRIVE_ROOT = ("/content/drive/MyDrive/projects/AI-DCC/mission1-data" if IS_DRIVE
              else os.path.join(os.path.expanduser("~"), "Downloads", "mission1-data"))

# 실제 폴더 구조 확인 결과 반영 (Training/Validation 각각 원천데이터=오디오, 라벨링데이터=json)
CONFIG = {
    "train_audio_root": os.path.join(DRIVE_ROOT, "Training", "1.원천데이터", "TS_서울_구급"),
    "train_label_root": os.path.join(DRIVE_ROOT, "Training", "2.라벨링데이터", "TL_서울_구급"),
    "val_audio_root":   os.path.join(DRIVE_ROOT, "Validation", "1.원천데이터", "VS_서울_구급"),
    "val_label_root":   os.path.join(DRIVE_ROOT, "Validation", "2.라벨링데이터", "VL_서울_구급"),

    "local_json_dir": "/content/local_json",     # JSON 로컬 복사본 (Drive I/O 병목 회피)
    "local_audio_dir": "/content/local_audio",   # 실제 필요한 오디오만 선택 복사
    "mel_cache_dir": "/content/mel_cache",
    "checkpoint_dir": "/content/checkpoints",

    "sample_rate": None,          # 다음 셀에서 실제 파일로 확인 후 채움
    "n_mels": 128,
    "n_fft": 1024,
    "hop_length": 512,
    "target_duration_sec": None,  # EDA 셀에서 train 기준으로만 결정

    "batch_size": 32,
    "epochs": 30,
    "lr_finetune": 1e-4,     # EfficientNet fine-tuning용 학습률
    "early_stop_patience": 5,
}
os.makedirs(CONFIG["mel_cache_dir"], exist_ok=True)
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)


Mounted at /content/drive
device=cuda, GPU=NVIDIA A100-SXM4-40GB


### 2. 실제 샘플레이트 확인

In [3]:
# ===== 원본 오디오의 실제 샘플레이트 확인 (전화 음성은 8000Hz인 경우도 많아 가정하지 않고 직접 확인) =====
def find_one_wav(root):
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.endswith(".wav"):
                return os.path.join(dirpath, f)
    return None

sample_wav = find_one_wav(CONFIG["train_audio_root"])
if sample_wav is None:
    raise FileNotFoundError(
        f"wav 파일을 찾지 못했습니다. train_audio_root 경로를 확인하세요: {CONFIG['train_audio_root']}"
    )

info = sf.info(sample_wav)
print(f"샘플 파일: {sample_wav}")
print(f"샘플레이트: {info.samplerate}Hz, 채널: {info.channels}, 길이: {info.duration:.1f}초")
CONFIG["sample_rate"] = info.samplerate


샘플 파일: /content/drive/MyDrive/projects/AI-DCC/mission1-data/Training/1.원천데이터/TS_서울_구급/651e553ba8f14bddba4c1312_20220603.wav
샘플레이트: 8000Hz, 채널: 1, 길이: 142.9초


### 3. 라벨링(JSON) 데이터 로컬 복사

In [4]:
# ===== JSON 전체를 로컬 디스크로 복사 (오디오보다 훨씬 가벼워서 통짜 복사가 빠름) =====
# Drive에 마운트된 상태로 JSON 수만 개를 하나씩 열면 매우 느리므로, 먼저 로컬로 복사한 뒤 파싱한다.
def copy_json_tree(src_root, dst_root):
    if os.path.exists(dst_root):
        print(f"이미 복사됨, 스킵: {dst_root}")
        return
    shutil.copytree(src_root, dst_root)
    print(f"복사 완료: {src_root} -> {dst_root}")

local_train_label = os.path.join(CONFIG["local_json_dir"], "train")
local_val_label = os.path.join(CONFIG["local_json_dir"], "val")

copy_json_tree(CONFIG["train_label_root"], local_train_label)
copy_json_tree(CONFIG["val_label_root"], local_val_label)


복사 완료: /content/drive/MyDrive/projects/AI-DCC/mission1-data/Training/2.라벨링데이터/TL_서울_구급 -> /content/local_json/train
복사 완료: /content/drive/MyDrive/projects/AI-DCC/mission1-data/Validation/2.라벨링데이터/VL_서울_구급 -> /content/local_json/val


### 4. Manifest 생성 (신고자 발화 + 성별 라벨)

In [5]:
# ===== 오디오 파일 인덱스 구축 후 _id 기준 매칭 =====
# audioPath 필드는 AI-hub 원본 경로라 실제 배포본(재가공됨)과 다름 -> 실제 파일명(<_id>_<날짜>.wav)으로 매칭

def build_audio_index(audio_root):
    """오디오 폴더를 스캔해 파일명 앞부분(ID)을 key로 하는 조회 테이블 생성"""
    index = {}
    for dirpath, _, filenames in os.walk(audio_root):
        for fname in filenames:
            if not fname.lower().endswith(".wav"):
                continue
            file_id = os.path.splitext(fname)[0].split("_")[0]
            index[file_id] = os.path.join(dirpath, fname)
    return index


def parse_one_json(json_path, audio_index):
    with open(json_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    file_id = meta.get("_id") or meta.get("recordId")
    call_id = meta.get("recordId") or file_id or os.path.splitext(os.path.basename(json_path))[0]
    gender = meta.get("gender")
    utterances = meta.get("utterances", meta.get("utterences", []))

    audio_path = audio_index.get(file_id)
    if audio_path is None:
        return []  # 매칭 실패 -> build_manifest에서 카운트로 확인

    records = []
    for utt in utterances:
        if utt.get("speaker") != 1:
            continue
        records.append({
            "call_id": call_id,
            "audio_path": audio_path,
            "start_sec": utt.get("startAt", 0) / 1000.0,
            "end_sec": utt.get("endAt", 0) / 1000.0,
            "gender": gender,
        })
    return records


def build_manifest(local_label_root, audio_root):
    audio_index = build_audio_index(audio_root)
    print(f"오디오 인덱스: {len(audio_index)}개 파일")

    records, matched, total = [], 0, 0
    for dirpath, _, filenames in os.walk(local_label_root):
        for fname in filenames:
            if not fname.endswith(".json"):
                continue
            total += 1
            new_records = parse_one_json(os.path.join(dirpath, fname), audio_index)
            if new_records:
                matched += 1
            records.extend(new_records)
    print(f"매칭된 콜: {matched}/{total}")
    return pd.DataFrame(records)


train_manifest = build_manifest(local_train_label, CONFIG["train_audio_root"])
val_manifest = build_manifest(local_val_label, CONFIG["val_audio_root"])

print(f"\ntrain: {len(train_manifest)}건 ({train_manifest['call_id'].nunique()} calls)")
print(f"val:   {len(val_manifest)}건 ({val_manifest['call_id'].nunique()} calls)")

sample_path = train_manifest["audio_path"].iloc[0]
print(f"\n경로 검증 샘플: {sample_path}")
print("파일 존재 여부:", os.path.exists(sample_path))

오디오 인덱스: 29200개 파일
매칭된 콜: 29200/29200
오디오 인덱스: 3640개 파일
매칭된 콜: 3640/3640

train: 462197건 (29200 calls)
val:   58102건 (3640 calls)

경로 검증 샘플: /content/drive/MyDrive/projects/AI-DCC/mission1-data/Training/1.원천데이터/TS_서울_구급/651e4bc5df6bf0d825aa5365_20220505.wav
파일 존재 여부: True


### 5. 발화 길이 분포 -> target_duration 결정

In [6]:
# ===== target_duration_sec 결정 (train만 사용, validation 정보로 결정하지 않음) =====
durations = train_manifest["end_sec"] - train_manifest["start_sec"]
print(durations.describe())

CONFIG["target_duration_sec"] = round(float(durations.quantile(0.95)), 2)
print(f"target_duration_sec = {CONFIG['target_duration_sec']}초 (95백분위수 기준)")


count    462197.000000
mean          2.119039
std           1.827057
min           0.009000
25%           0.809000
50%           1.490000
75%           2.826000
max          24.172000
dtype: float64
target_duration_sec = 5.97초 (95백분위수 기준)


### 6. 필요한 오디오만 로컬로 선택 복사

In [7]:
# ===== manifest에 등장하는 오디오만 로컬 디스크로 복사 (전체 데이터셋 복사 금지) =====
def copy_needed_audio(manifest_df, local_dir):
    os.makedirs(local_dir, exist_ok=True)
    path_map, missing = {}, 0
    for src_path in manifest_df["audio_path"].dropna().unique():
        dst_path = os.path.join(local_dir, os.path.basename(src_path))
        if os.path.exists(dst_path):
            if os.path.exists(src_path) and os.path.getsize(dst_path) == os.path.getsize(src_path):
                path_map[src_path] = dst_path
                continue
            os.remove(dst_path)  # 크기 불일치 = 중간에 끊긴 파일 -> 삭제 후 재복사
        if not os.path.exists(src_path):
            missing += 1
            continue
        tmp_path = dst_path + ".part"
        shutil.copy(src_path, tmp_path)
        os.rename(tmp_path, dst_path)  # 원자적 복사 -> 중단돼도 손상 파일 안 남음
        path_map[src_path] = dst_path
    if missing:
        print(f"⚠️ 원본을 찾지 못한 파일 {missing}건")
    return path_map

train_path_map = copy_needed_audio(train_manifest, os.path.join(CONFIG["local_audio_dir"], "train"))
val_path_map = copy_needed_audio(val_manifest, os.path.join(CONFIG["local_audio_dir"], "val"))

train_manifest["audio_path"] = train_manifest["audio_path"].map(train_path_map)
val_manifest["audio_path"] = val_manifest["audio_path"].map(val_path_map)

# 복사 실패로 audio_path가 NaN이 된 행 제거
train_manifest = train_manifest.dropna(subset=["audio_path"]).reset_index(drop=True)
val_manifest = val_manifest.dropna(subset=["audio_path"]).reset_index(drop=True)
print(f"최종 train: {len(train_manifest)}건, val: {len(val_manifest)}건")


KeyboardInterrupt: 

### 7. Mel-Spectrogram 추출

In [ ]:
# ===== log-mel spectrogram 추출 (split별 로컬 캐싱) =====
def extract_mel_spectrogram(audio_path, start_sec, end_sec, call_id, seg_idx, config, split):
    cache_path = os.path.join(config["mel_cache_dir"], f"{split}_{call_id}_{seg_idx}.npy")
    if os.path.exists(cache_path):
        return np.load(cache_path)

    sr = config["sample_rate"]
    y, _ = librosa.load(audio_path, sr=sr, offset=start_sec, duration=end_sec - start_sec)

    target_len = int(config["target_duration_sec"] * sr)
    y = np.pad(y, (0, max(0, target_len - len(y))))[:target_len]

    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=config["n_mels"], n_fft=config["n_fft"], hop_length=config["hop_length"]
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)  # 샘플별 정규화

    np.save(cache_path, mel_db.astype(np.float32))
    return mel_db.astype(np.float32)


### 8. Dataset / DataLoader

In [ ]:
# ===== PyTorch Dataset =====
class GenderDataset(Dataset):
    def __init__(self, df, config, split):
        self.df = df.reset_index(drop=True)
        self.config = config
        self.split = split
        self.label_map = {"M": 0, "F": 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mel = extract_mel_spectrogram(row["audio_path"], row["start_sec"], row["end_sec"],
                                       row["call_id"], idx, self.config, self.split)
        return torch.tensor(mel).unsqueeze(0), self.label_map[row["gender"]], row["call_id"]


train_loader = DataLoader(GenderDataset(train_manifest, CONFIG, "train"),
                           batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2)
val_loader = DataLoader(GenderDataset(val_manifest, CONFIG, "val"),
                         batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2)


### 9. 모델 정의 (EfficientNet)

In [ ]:
# ===== 담당 backbone: pretrained EfficientNet-B0 (FAQ Q1 기준 공개 pretrained 모델 사용 허용) =====
class EfficientNetGender(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        self.backbone = models.efficientnet_b0(weights=weights)

        # mel-spectrogram은 1채널이므로 원래 3채널(RGB)용 첫 conv를 교체
        old_conv = self.backbone.features[0][0]
        new_conv = nn.Conv2d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                              stride=old_conv.stride, padding=old_conv.padding, bias=False)
        if pretrained:
            new_conv.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
        self.backbone.features[0][0] = new_conv

        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier[1] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)


### 10. 학습 함수 (모델 비교를 위해 재사용 가능하도록 함수화)

In [ ]:
# ===== 학습 루프: 모델/실행이름을 인자로 받아 재사용 (체크포인트 파일명도 실행별로 분리) =====
def train_model(model, run_name, config, train_loader, val_loader, lr):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    def run_epoch(loader, train):
        model.train() if train else model.eval()
        total_loss, preds, labels, calls = 0, [], [], []
        with torch.set_grad_enabled(train):
            for mel, label, call_id in loader:
                mel, label = mel.to(device), label.to(device)
                logits = model(mel)
                loss = criterion(logits, label)
                if train:
                    optimizer.zero_grad(); loss.backward(); optimizer.step()
                total_loss += loss.item() * mel.size(0)
                preds.extend(logits.argmax(1).cpu().tolist())
                labels.extend(label.cpu().tolist())
                calls.extend(call_id)
        return total_loss / len(loader.dataset), preds, labels, calls

    best_f1, patience_counter = 0.0, 0
    ckpt_path = os.path.join(config["checkpoint_dir"], f"{run_name}_best.pt")
    history = []  # epoch별 Accuracy/F1 기록 (맨 마지막 그래프 셀에서 사용)

    for epoch in range(config["epochs"]):
        train_loss, _, _, _ = run_epoch(train_loader, train=True)
        val_loss, val_preds, val_labels, val_calls = run_epoch(val_loader, train=False)

        # 통화 단위 다수결 집계 (팀 공통 평가 기준)
        agg = pd.DataFrame({"call_id": val_calls, "pred": val_preds, "true": val_labels}) \
                .groupby("call_id").agg(pred=("pred", lambda x: x.mode()[0]), true=("true", "first"))
        call_acc = accuracy_score(agg["true"], agg["pred"])
        call_f1 = f1_score(agg["true"], agg["pred"])
        history.append({"epoch": epoch + 1, "val_acc": call_acc, "val_f1": call_f1})

        print(f"[{run_name}] epoch {epoch+1} train_loss={train_loss:.4f} "
              f"val_loss={val_loss:.4f} call_acc={call_acc:.4f} call_f1={call_f1:.4f}")

        if call_f1 > best_f1:
            best_f1, patience_counter = call_f1, 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            patience_counter += 1
            if patience_counter >= config["early_stop_patience"]:
                print(f"[{run_name}] early stopping (patience={config['early_stop_patience']})")
                break

    return best_f1, history


### 11. 학습 실행

In [ ]:
# ===== EfficientNet 학습 실행 =====
best_f1, history = train_model(
    EfficientNetGender(pretrained=True), "efficientnet_b0", CONFIG, train_loader, val_loader, lr=CONFIG["lr_finetune"]
)
print(f"\nEfficientNet-B0 best val F1: {best_f1:.4f}")


### 12. 추론 스크립트 생성 (inference.py) — 실행규칙 준수

실행규칙: `python inference.py --audio_dir ... --label_dir ... --ckpt_path ... --output ...` 형태로 **단일 명령어 실행**이 가능해야 하고, 출력 CSV는 `[audio file name], [gender]` 형식이어야 함. 아래 셀은 노트북 상태와 무관하게 독립적으로 동작하는 `inference.py`를 실제로 파일로 생성한다.

In [ ]:
%%writefile inference.py
"""
Mission 1 추론 스크립트 - 신고자 성별 분류

사용법:
    python inference.py --audio_dir {wav folder} --label_dir {json folder} \
        --ckpt_path {checkpoint file} --output ./outputs/mission1.csv

출력 CSV 컬럼: [audio file name], [gender]
"""
import os
import json
import argparse
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torchvision.models as models

# ===== 학습 시 사용한 값과 반드시 동일해야 함 (이 노트북의 CONFIG 기준) =====
SAMPLE_RATE = 8000
TARGET_DURATION_SEC = 5.97
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 512


class EfficientNetGender(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        # 추론 시엔 저장된 가중치로 전부 덮어쓸 것이므로 ImageNet pretrained 다운로드 불필요
        self.backbone = models.efficientnet_b0(weights=None)
        old_conv = self.backbone.features[0][0]
        self.backbone.features[0][0] = nn.Conv2d(
            1, old_conv.out_channels, kernel_size=old_conv.kernel_size,
            stride=old_conv.stride, padding=old_conv.padding, bias=False
        )
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier[1] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)


def build_audio_index(audio_dir):
    """오디오 파일명(<id>_<날짜>.wav)에서 ID를 추출해 조회 테이블 생성"""
    index = {}
    for dirpath, _, filenames in os.walk(audio_dir):
        for fname in filenames:
            if not fname.lower().endswith(".wav"):
                continue
            file_id = os.path.splitext(fname)[0].split("_")[0]
            index[file_id] = os.path.join(dirpath, fname)
    return index


def extract_mel_spectrogram(audio_path, start_sec, end_sec):
    y, _ = librosa.load(audio_path, sr=SAMPLE_RATE, offset=start_sec, duration=end_sec - start_sec)
    target_len = int(TARGET_DURATION_SEC * SAMPLE_RATE)
    y = np.pad(y, (0, max(0, target_len - len(y))))[:target_len]
    mel = librosa.feature.melspectrogram(y=y, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    return mel_db.astype(np.float32)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--audio_dir", required=True, help="원천 오디오(wav) 폴더")
    parser.add_argument("--label_dir", required=True, help="라벨링(json) 폴더")
    parser.add_argument("--ckpt_path", required=True, help="학습된 체크포인트(.pt) 경로")
    parser.add_argument("--output", required=True, help="결과 CSV 저장 경로")
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = EfficientNetGender()
    model.load_state_dict(torch.load(args.ckpt_path, map_location=device))
    model.to(device)
    model.eval()

    audio_index = build_audio_index(args.audio_dir)
    inv_label_map = {0: "M", 1: "F"}
    rows = []

    for dirpath, _, filenames in os.walk(args.label_dir):
        for fname in filenames:
            if not fname.endswith(".json"):
                continue
            with open(os.path.join(dirpath, fname), "r", encoding="utf-8") as f:
                meta = json.load(f)

            file_id = meta.get("_id") or meta.get("recordId")
            audio_path = audio_index.get(file_id)
            if audio_path is None:
                continue

            utterances = meta.get("utterances", meta.get("utterences", []))
            reporter_segs = [u for u in utterances if u.get("speaker") == 1]
            if not reporter_segs:
                reporter_segs = utterances  # 화자 구분 정보가 없는 경우 전체 발화로 대체

            preds = []
            for utt in reporter_segs:
                start_sec = utt.get("startAt", 0) / 1000.0
                end_sec = utt.get("endAt", 0) / 1000.0
                if end_sec <= start_sec:
                    continue
                mel = extract_mel_spectrogram(audio_path, start_sec, end_sec)
                x = torch.tensor(mel).unsqueeze(0).unsqueeze(0).to(device)
                with torch.no_grad():
                    logit = model(x)
                preds.append(logit.argmax(1).item())

            if not preds:
                continue
            majority = max(set(preds), key=preds.count)  # 오디오 파일 단위 다수결
            rows.append({
                "audio file name": os.path.basename(audio_path),
                "gender": inv_label_map[majority],
            })

    out_dir = os.path.dirname(args.output)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    pd.DataFrame(rows).to_csv(args.output, index=False)
    print(f"저장 완료: {args.output} ({len(rows)}건)")


if __name__ == "__main__":
    main()


In [ ]:
# ===== 생성된 inference.py 실제 실행 테스트 (제출 전 검증용) =====
import subprocess

ckpt_path = os.path.join(CONFIG["checkpoint_dir"], "efficientnet_b0_best.pt")  # 제출할 체크포인트로 교체
cmd = [
    "python", "inference.py",
    "--audio_dir", CONFIG["val_audio_root"],
    "--label_dir", CONFIG["val_label_root"],
    "--ckpt_path", ckpt_path,
    "--output", "./outputs/mission1.csv",
]
subprocess.run(cmd, check=True)

pd.read_csv("./outputs/mission1.csv").head()


### 13. 성능 그래프 (Accuracy / F1-Score)

In [ ]:
# ===== Epoch별 Validation Accuracy / F1-Score 그래프 =====
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in history]
accs = [h["val_acc"] for h in history]
f1s = [h["val_f1"] for h in history]

plt.figure(figsize=(8, 4.5))
plt.plot(epochs, accs, marker="o", label="Accuracy")
plt.plot(epochs, f1s, marker="s", label="F1-Score")
plt.xlabel("Epoch")
plt.ylabel("Score (call-level)")
plt.title("Mission 1 - EfficientNet-B0 Validation Accuracy / F1-Score by Epoch")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("/content/mission1_metrics.png", dpi=150)
plt.show()
